In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# The Mars Rover Environment\n",
    "\n",
    "## Introduction2RL_2025 Course\n",
    "\n",
    "**Author**: Simon Hirlaender  \n",
    "**Date**: Spring 2025  \n",
    "**Institution**: Paris Lodron University Salzburg\n",
    "\n",
    "---\n",
    "\n",
    "This notebook provides a detailed introduction to our custom Mars Rover environment, which we'll use throughout this course to demonstrate reinforcement learning concepts."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Setup\n",
    "\n",
    "First, let's import the necessary libraries and our custom environment."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import gymnasium as gym\n",
    "import sys\n",
    "\n",
    "# Add the parent directory to the path so we can import our environment\n",
    "sys.path.append('..')\n",
    "from environments.mars_rover import MarsRoverEnv, MarsRoverMRPWrapper"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Environment Overview\n",
    "\n",
    "The Mars Rover environment simulates a rover navigating a linear Martian landscape. The goal is to reach the right terminal state while avoiding the left terminal state.\n",
    "\n",
    "### Key Features:\n",
    "\n",
    "- **Discrete States**: The environment consists of a linear array of states\n",
    "- **Terminal States**: The leftmost and rightmost states are terminal\n",
    "- **Probabilistic Transitions**: Actions have uncertain outcomes\n",
    "- **Sparse Rewards**: Rewards are only given upon reaching terminal states\n",
    "\n",
    "Let's create an instance of our environment and explore its properties."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create the environment with default parameters\n",
    "env = MarsRoverEnv()\n",
    "\n",
    "# Display basic information\n",
    "print(f\"Observation space: {env.observation_space}\")\n",
    "print(f\"Action space: {env.action_space}\")\n",
    "print(f\"Number of states: {env.nS}\")\n",
    "print(f\"Number of actions: {env.nA}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Environment Parameters\n",
    "\n",
    "Our Mars Rover environment is highly customizable. Let's explore the parameters we can adjust:"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create a more complex environment\n",
    "complex_env = MarsRoverEnv(\n",
    "    n_states=10,           # 10 non-terminal states (12 total including terminals)\n",
    "    p_stay=0.3,            # 30% chance of staying in place\n",
    "    p_backward=0.2,        # 20% chance of moving in the opposite direction\n",
    "    left_side_reward=-1.0, # Negative reward for reaching left terminal\n",
    "    right_side_reward=2.0  # Higher reward for reaching right terminal\n",
    ")\n",
    "\n",
    "print(f\"Number of states: {complex_env.nS}\")\n",
    "print(f\"Probability of staying in place: {complex_env.p_stay}\")\n",
    "print(f\"Probability of moving backward: {complex_env.p_backward}\")\n",
    "print(f\"Probability of moving forward: {1 - complex_env.p_stay - complex_env.p_backward}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Visualizing the Environment\n",
    "\n",
    "Let's create a visualization of our Mars Rover environment to better understand its structure."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def visualize_mars_rover_env(env):\n",
    "    \"\"\"Visualize the Mars Rover environment.\"\"\"\n",
    "    # Create a figure\n",
    "    fig, ax = plt.subplots(figsize=(12, 4))\n",
    "    \n",
    "    # Draw states\n",
    "    for i in range(env.nS):\n",
    "        if i == 0:  # Left terminal\n",
    "            color = 'red'\n",
    "            label = f\"Terminal\\nReward: {env.left_side_reward}\"\n",
    "        elif i == env.nS - 1:  # Right terminal\n",
    "            color = 'green'\n",
    "            label = f\"Terminal\\nReward: {env.right_side_reward}\"\n",
    "        else:  # Non-terminal\n",
    "            color = 'lightblue'\n",
    "            label = f\"State {i}\"\n",
    "        \n",
    "        # Draw state circle\n",
    "        circle = plt.Circle((i, 0), 0.4, color=color, alpha=0.7)\n",
    "        ax.add_patch(circle)\n",
    "        ax.text(i, 0, label, ha='center', va='center', fontsize=9)\n",
    "    \n",
    "    # Draw transition probabilities for a middle state\n",
    "    mid_state = env.nS // 2\n",
    "    \n",
    "    # Right action transitions\n",
    "    ax.arrow(mid_state, 0.5, 1, 0, head_width=0.1, head_length=0.1, fc='blue', ec='blue')\n",
    "    ax.text(mid_state + 0.5, 0.7, f\"P(forward) = {1-env.p_stay-env.p_backward:.1f}\", color='blue', ha='center')\n",
    "    \n",
    "    ax.arrow(mid_state, 0.8, 0, 0.4, head_width=0.1, head_length=0.1, fc='orange', ec='orange')\n",
    "    ax.text(mid_state, 1.3, f\"P(stay) = {env.p_stay:.1f}\", color='orange', ha='center')\n",
    "    \n",
    "    ax.arrow(mid_state, 0.5, -1, 0, head_width=0.1, head_length=0.1, fc='red', ec='red')\n",
    "    ax.text(mid_state - 0.5, 0.7, f\"P(backward) = {env.p_backward:.1f}\", color='red', ha='center')\n",
    "    \n",
    "    # Set limits and labels\n",
    "    ax.set_xlim(-1, env.nS)\n",
    "    ax.set_ylim(-1, 2)\n",
    "    ax.set_title(f\"Mars Rover Environment (n_states={env.nS-2})\")\n",
    "    ax.set_xticks(range(env.nS))\n",
    "    ax.set_xticklabels([f\"{i}\" for i in range(env.nS)])\n",
    "    ax.set_yticks([])\n",
    "    ax.set_aspect('equal')\n",
    "    ax.grid(True, linestyle='--', alpha=0.6)\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    return fig\n",
    "\n",
    "# Visualize our environment\n",
    "fig = visualize_mars_rover_env(env)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Interacting with the Environment\n",
    "\n",
    "Let's run a few episodes with random actions to see how the environment behaves."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def run_episode(env, max_steps=100, render=True):\n",
    "    \"\"\"Run a single episode with random actions.\"\"\"\n",
    "    # Reset environment\n",
    "    state, _ = env.reset()\n",
    "    total_reward = 0\n",
    "    steps = 0\n",
    "    done = False\n",
    "    \n",
    "    # Store trajectory\n",
    "    trajectory = [(state, None, 0, False)]\n",
    "    \n",
    "    # Run episode\n",
    "    while not done and steps < max_steps:\n",
    "        # Choose random action\n",
    "        action = env.action_space.sample()\n",
    "        \n",
    "        # Take step\n",
    "        next_state, reward, terminated, truncated, _ = env.step(action)\n",
    "        done = terminated or truncated\n",
    "        \n",
    "        # Update\n",
    "        total_reward += reward\n",
    "        steps += 1\n",
    "        \n",
    "        # Store step in trajectory\n",
    "        trajectory.append((next_state, action, reward, done))\n",
    "        \n",
    "        # Update state\n",
    "        state = next_state\n",
    "        \n",
    "        # Render if requested\n",
    "        if render:\n",
    "            env.render()\n",
    "    \n",
    "    return trajectory, total_reward, steps\n",
    "\n",
    "# Run and analyze a few random episodes\n",
    "n_episodes = 5\n",
    "results = []\n",
    "\n",
    "for i in range(n_episodes):\n",
    "    trajectory, total_reward, steps = run_episode(env, render=False)\n",
    "    results.append((total_reward, steps))\n",
    "    \n",
    "    print(f\"Episode {i+1}: Steps={steps}, Total Reward={total_reward}\")\n",
    "    \n",
    "    # Print trajectory summary\n",
    "    states = [step[0] for step in trajectory]\n",
    "    print(f\"State sequence: {states}\")\n",
    "    print(\"-\" * 40)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Analyzing Episode Statistics\n",
    "\n",
    "Let's run more episodes and analyze the statistics to understand the environment better."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Run many episodes\n",
    "n_episodes = 1000\n",
    "rewards = []\n",
    "steps_list = []\n",
    "terminal_states = []\n",
    "\n",
    "for _ in range(n_episodes):\n",
    "    trajectory, total_reward, steps = run_episode(env, render=False)\n",
    "    rewards.append(total_reward)\n",
    "    steps_list.append(steps)\n",
    "    terminal_states.append(trajectory[-1][0])  # Last state in trajectory\n",
    "\n",
    "# Plot statistics\n",
    "fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))\n",
    "\n",
    "# Rewards histogram\n",
    "sns.histplot(rewards, bins=2, ax=ax1)\n",
    "ax1.set_title(f\"Reward Distribution (n={n_episodes})\")\n",
    "ax1.set_xlabel(\"Total Reward\")\n",
    "ax1.set_ylabel(\"Frequency\")\n",
    "\n",
    "# Steps histogram\n",
    "sns.histplot(steps_list, bins=20, ax=ax2)\n",
    "ax2.set_title(f\"Steps Distribution (n={n_episodes})\")\n",
    "ax2.set_xlabel(\"Steps\")\n",
    "ax2.set_ylabel(\"Frequency\")\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "# Print summary statistics\n",
    "print(f\"Average reward: {np.mean(rewards):.2f}\")\n",
    "print(f\"Average steps: {np.mean(steps_list):.2f}\")\n",
    "print(f\"Reached left terminal: {terminal_states.count(0)} times ({terminal_states.count(0)/n_episodes:.1%})\")\n",
    "print(f\"Reached right terminal: {terminal_states.count(env.nS-1)} times ({terminal_states.count(env.nS-1)/n_episodes:.1%})\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Using the MRP Wrapper\n",
    "\n",
    "The `MarsRoverMRPWrapper` converts our MDP into a Markov Reward Process by fixing a policy. This is useful for understanding the behavior of a specific policy."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create a simple policy: always move right\n",
    "always_right_policy = np.zeros((env.nS, env.nA))\n",
    "always_right_policy[:, MarsRoverEnv.RIGHT] = 1.0\n",
    "\n",
    "# Create MRP wrapper with this policy\n",
    "mrp_env = MarsRoverMRPWrapper(env, policy=always_right_policy)\n",
    "\n",
    "# Run episodes and analyze results\n",
    "n_episodes = 100\n",
    "mrp_rewards = []\n",
    "mrp_steps = []\n",
    "\n",
    "for _ in range(n_episodes):\n",
    "    trajectory, total_reward, steps = run_episode(mrp_env, render=False)\n",
    "    mrp_rewards.append(total_reward)\n",
    "    mrp_steps.append(steps)\n",
    "\n",
    "print(f\"MRP (Always Right Policy)\")\n",
    "print(f\"Average reward: {np.mean(mrp_rewards):.2f}\")\n",
    "print(f\"Average steps: {np.mean(mrp_steps):.2f}\")\n",
    "\n",
    "# Compare with random policy\n",
    "print(\"\\nRandom Policy (from earlier)\")\n",
    "print(f\"Average reward: {np.mean(rewards):.2f}\")\n",
    "print(f\"Average steps: {np.mean(steps_list):.2f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Understanding Transition Probabilities\n",
    "\n",
    "Let's examine the transition probabilities in more detail to understand the environment dynamics."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def visualize_transitions(env, state):\n",
    "    \"\"\"Visualize transition probabilities from a given state.\"\"\"\n",
    "    # Get transitions for left and right actions\n",
    "    left_transitions = env.P[state][MarsRoverEnv.LEFT]\n",
    "    right_transitions = env.P[state][MarsRoverEnv.RIGHT]\n",
    "    \n",
    "    # Create a figure\n",
    "    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))\n",
    "    \n",
    "    # Left action\n",
    "    states_left = [t[1] for t in left_transitions]\n",
    "    probs_left = [t[0] for t in left_transitions]\n",
    "    rewards_left = [t[2] for t in left_transitions]\n",
    "    done_left = [t[3] for t in left_transitions]\n",
    "    \n",
    "    ax1.bar(states_left, probs_left, color='skyblue')\n",
    "    ax1.set_title(f\"Transition Probabilities from State {state} (LEFT Action)\")\n",
    "    ax1.set_xlabel(\"Next State\")\n",
    "    ax1.set_ylabel(\"Probability\")\n",
    "    ax1.set_xticks(range(env.nS))\n",
    "    ax1.grid(True, linestyle='--', alpha=0.6)\n",
    "    \n",
    "    # Add reward and terminal annotations\n",
    "    for i, (s, p, r, d) in enumerate(zip(states_left, probs_left, rewards_left, done_left)):\n",
    "        label = f\"R={r:.1f}\\nDone={d}\"\n",
    "        ax1.annotate(label, (s, p), textcoords=\"offset points\", xytext=(0,5), ha='center')\n",
    "    \n",
    "    # Right action\n",
    "    states_right = [t[1] for t in right_transitions]\n",
    "    probs_right = [t[0] for t in right_transitions]\n",
    "    rewards_right = [t[2] for t in right_transitions]\n",
    "    done_right = [t[3] for t in right_transitions]\n",
    "    \n",
    "    ax2.bar(states_right, probs_right, color='salmon')\n",
    "    ax2.set_title(f\"Transition Probabilities from State {state} (RIGHT Action)\")\n",
    "    ax2.set_xlabel(\"Next State\")\n",
    "    ax2.set_ylabel(\"Probability\")\n",
    "    ax2.set_xticks(range(env.nS))\n",
    "    ax2.grid(True, linestyle='--', alpha=0.6)\n",
    "    \n",
    "    # Add reward and terminal annotations\n",
    "    for i, (s, p, r, d) in enumerate(zip(states_right, probs_right, rewards_right, done_right)):\n",
    "        label = f\"R={r:.1f}\\nDone={d}\"\n",
    "        ax2.annotate(label, (s, p), textcoords=\"offset points\", xytext=(0,5), ha='center')\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    return fig\n",
    "\n",
    "# Visualize transitions from middle state\n",
    "middle_state = env.nS // 2\n",
    "visualize_transitions(env, middle_state)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Conclusion\n",
    "\n",
    "In this notebook, we've explored the Mars Rover environment that we'll use throughout this course. We've seen how to:\n",
    "\n",
    "1. Create and customize the environment\n",
    "2. Visualize its structure and dynamics\n",
    "3. Run episodes and analyze results\n",
    "4. Convert the MDP to an MRP using a fixed policy\n",
    "5. Examine transition probabilities in detail\n",
    "\n",
    "In the next notebooks, we'll use this environment to implement and test various reinforcement learning algorithms, starting with dynamic programming methods for solving MDPs."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.12.6"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}